In [32]:

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, pandas as pd, time, json
from pathlib import Path
from scipy import stats
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import argparse

from step0_model import create_hybrid_mlp_model
from utils_SIR import create_dataloaders, get_device, EarlyStopping, compute_metrics


In [33]:
# Reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

# ───────────────────────────────────────────────
# Loss + per-compartment MAE
def compute_balanced_loss(pred, target, device):
    S_pred, I_pred, R_pred = pred[:, :, 0], pred[:, :, 1], pred[:, :, 2]
    S_true, I_true, R_true = target[:, :, 0], target[:, :, 1], target[:, :, 2]

    # squared losses
    loss_S = (S_pred - S_true).pow(2).mean()
    loss_I = (I_pred - I_true).pow(2).mean()
    I_peak_loss = (I_pred.max(dim=1)[0] - I_true.max(dim=1)[0]).pow(2).mean()
    I_mean_loss = ((I_pred.mean(dim=1) - I_true.mean(dim=1))**2).mean()

    total_loss = 15*loss_S + 15*loss_I + 8*I_peak_loss + 3*I_mean_loss

    # compartment MAE
    mae_S, mae_I, mae_R = ((S_pred - S_true).abs().mean(),
                           (I_pred - I_true).abs().mean(),
                           (R_pred - R_true).abs().mean())

    return total_loss, mae_S.item(), mae_I.item(), mae_R.item()

# ───────────────────────────────────────────────
# R² per compartment
def r2_score_comp(pred, true):
    return 1 - ((pred-true)**2).sum(dim=1)/((true-true.mean(dim=1, keepdim=True))**2).sum(dim=1)


In [34]:
# ───────────────────────────────────────────────
# Train / Validate epoch
def run_epoch(model, loader, optimizer=None, device='cpu', n_timesteps=200):
    training = optimizer is not None
    if training: model.train()
    else: model.eval()

    total_loss = total_mae_S = total_mae_I = total_mae_R = 0
    all_pred, all_true = [], []

    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = batch.to(device)
            if training: optimizer.zero_grad()

            pred = model(batch, n_timesteps=n_timesteps)
            target = batch.y
            loss, mae_S, mae_I, mae_R = compute_balanced_loss(pred, target, device)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss   += loss.item()
            total_mae_S  += mae_S
            total_mae_I  += mae_I
            total_mae_R  += mae_R

            all_pred.append(pred.detach().cpu())
            all_true.append(target.detach().cpu())

    n_batches = len(loader)
    preds = torch.cat(all_pred, dim=0)
    trues = torch.cat(all_true, dim=0)

    metrics = {
        'loss': total_loss/n_batches,
        'MAE_S': total_mae_S/n_batches,
        'MAE_I': total_mae_I/n_batches,
        'MAE_R': total_mae_R/n_batches,
        'R2_S': r2_score_comp(preds[:, :, 0], trues[:, :, 0]).mean().item(),
        'R2_I': r2_score_comp(preds[:, :, 1], trues[:, :, 1]).mean().item(),
        'R2_R': r2_score_comp(preds[:, :, 2], trues[:, :, 2]).mean().item(),
    }

    return metrics

In [35]:
# ───────────────────────────────────────────────
# Single replicate training
def train_replicate(rep_id, seed, config, dataloaders, output_dir, weight_mode='modest'):
    set_seed(seed)
    device = get_device()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    model = create_hybrid_mlp_model(config).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=1e-6)
    early_stopping = EarlyStopping(patience=config['patience'])

    history = {k: [] for k in ['train_loss','val_loss','MAE_S','MAE_I','MAE_R','R2_S','R2_I','R2_R']}
    best_val_r2 = -np.inf

    for epoch in range(config['epochs']):
        train_metrics = run_epoch(model, dataloaders['train'], optimizer, device, config['n_timepoints'])
        val_metrics   = run_epoch(model, dataloaders['val'], None, device, config['n_timepoints'])
        scheduler.step()

        # log history
        for k, v in zip(history.keys(), list(train_metrics.values()) + list(val_metrics.values())):
            history[k].append(v)

        # print live metrics
        print(f"Rep {rep_id} | Epoch {epoch+1}: Val Loss={val_metrics['loss']:.4f}, "
              f"MAE_S={val_metrics['MAE_S']:.2f}, MAE_I={val_metrics['MAE_I']:.2f}, "
              f"MAE_R={val_metrics['MAE_R']:.2f}, "
              f"R2_I={val_metrics['R2_I']:.4f}")

        # save best checkpoint
        val_r2_mean = np.mean([val_metrics['R2_S'], val_metrics['R2_I'], val_metrics['R2_R']])
        if val_r2_mean > best_val_r2:
            best_val_r2 = val_r2_mean
            torch.save(model.state_dict(), output_dir/f'best_model_{rep_id}.pt')

        if early_stopping(val_metrics['loss']):
            print(f"Early stopping at epoch {epoch+1}")
            break

    return {'replicate_id': rep_id, 'seed': seed, 'best_val_r2': best_val_r2}, history


In [36]:
# ───────────────────────────────────────────────
# Multiple replicates
def train_replicates(n_reps, seeds, config, dataloaders, output_dir):
    all_results, all_histories = [], []
    for i, seed in enumerate(seeds, 1):
        res, hist = train_replicate(i, seed, config, dataloaders, output_dir)
        all_results.append(res)
        all_histories.append(hist)
    return all_results, all_histories

# ───────────────────────────────────────────────
# Entry point
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', type=str, default='epidemic_data_age_adaptive_sobol_split_augmented.pkl')
    parser.add_argument('--output_dir', type=str, default='replicates_outputs')
    parser.add_argument('--n_replicates', type=int, default=2)
    parser.add_argument('--seeds', type=str, default=None)
    parser.add_argument('--epochs', type=int, default=50)
    parser.add_argument('--batch_size', type=int, default=35)
    parser.add_argument('--lr', type=float, default=0.01)
    parser.add_argument('--weight_decay', type=float, default=1e-3)
    parser.add_argument('--patience', type=int, default=35)
    args, _ = parser.parse_known_args()

    seeds = [int(s) for s in args.seeds.split(',')] if args.seeds else [42+i for i in range(args.n_replicates)]
    config = {'n_params':3,'n_fourier':64,'fusion_hidden':128,'latent_dim':64,'decoder_hidden':64,
              'dropout':0.3,'n_knots':12,'n_timepoints':200,'total_population':10000,
              'epochs':args.epochs,'batch_size':args.batch_size,'lr':args.lr,'weight_decay':args.weight_decay,
              'patience':args.patience}

    dataloaders = create_dataloaders(args.input, batch_size=config['batch_size'])
    all_results, all_histories = train_replicates(args.n_replicates, seeds, config, dataloaders, args.output_dir)

    # Save results
    df = pd.DataFrame(all_results)
    df.to_csv(Path(args.output_dir)/'replicates_results.csv', index=False)
    print(f"Saved replicates_results.csv in {args.output_dir}")


Loading dataset : epidemic_data_age_adaptive_sobol_split_augmented.pkl
  n_timepoints  : 200
  Train=6992, Val=374, Test=378

⚠  Using CPU (GPU not available)
Rep 1 | Epoch 1: Val Loss=203905.9308, MAE_S=63.90, MAE_I=2.80, MAE_R=62.07, R2_I=0.4340
Rep 1 | Epoch 2: Val Loss=183539.7741, MAE_S=62.80, MAE_I=2.71, MAE_R=61.15, R2_I=0.4448
Rep 1 | Epoch 3: Val Loss=157174.3884, MAE_S=60.47, MAE_I=2.80, MAE_R=58.66, R2_I=0.4486
Rep 1 | Epoch 4: Val Loss=162600.6709, MAE_S=65.18, MAE_I=2.57, MAE_R=64.02, R2_I=0.4962
Rep 1 | Epoch 5: Val Loss=164617.4359, MAE_S=67.66, MAE_I=3.32, MAE_R=69.64, R2_I=-0.2734
Rep 1 | Epoch 6: Val Loss=204959.0073, MAE_S=67.19, MAE_I=2.51, MAE_R=65.84, R2_I=0.5059
Rep 1 | Epoch 7: Val Loss=153385.8216, MAE_S=69.52, MAE_I=3.22, MAE_R=67.69, R2_I=0.1898
Rep 1 | Epoch 8: Val Loss=227821.4626, MAE_S=68.51, MAE_I=3.31, MAE_R=66.24, R2_I=0.1472
Rep 1 | Epoch 9: Val Loss=126387.2934, MAE_S=56.13, MAE_I=3.29, MAE_R=54.52, R2_I=0.1578
Rep 1 | Epoch 10: Val Loss=138673.1294